# Tutorial 3: Querying Historical Data via the REST API

This tutorial demonstrates how to query power quality data through the EQ gateway REST API,
which is backed by DuckDB for fast analytical queries over parquet files.

**What you will learn:**
1. Discover connected devices
2. Query PMon and CPOW data with time range filters
3. Run raw SQL queries for custom aggregations
4. Work with power quality events
5. List available data files

**Prerequisites:** A running EQ gateway (the API is served at `http://localhost:8080` by default).

**Response format:** Most data endpoints return [Arrow IPC](https://arrow.apache.org/docs/format/Columnar.html#ipc-streaming-format) binary streams, which `pyarrow` can decode directly into DataFrames with zero-copy efficiency.

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import requests

%matplotlib inline

from equser.api import GatewayClient

# Gateway URL - adjust if running remotely
GATEWAY = 'http://localhost:8080'
client = GatewayClient(GATEWAY)

## 2. Discover devices

The `/api/v1/devices` endpoint returns all registered sensor devices.

In [ ]:
devices = client.list_devices()
print(f"Found {len(devices)} device(s):")
for dev in devices:
    print(f"  {dev}")

In [ ]:
# Select a device ID to use for the rest of this tutorial.
# Replace with an ID from the list above if needed.
if devices:
    # Device objects may be dicts with 'id' key, or plain strings
    first = devices[0]
    device_id = first['id'] if isinstance(first, dict) else str(first)
else:
    device_id = 'wave-001'
print(f"Using device: {device_id}")

## 3. Query PMon data

The `/api/v1/devices/{id}/pmon/data` endpoint returns aggregated power metrics.
The response is Arrow IPC binary, which we decode into a pyarrow Table and then
convert to a pandas DataFrame for easy manipulation.

In [ ]:
# Fetch PMon data (optionally add start_time, end_time, metrics, limit params)
pmon_table = client.get_arrow(f'/api/v1/devices/{device_id}/pmon/data', params={'limit': 5000})
pmon_df = pmon_table.to_pandas()
print(f"Received {len(pmon_df)} rows")
print(f"Columns: {list(pmon_df.columns)}")
pmon_df.head()

In [ ]:
# Plot voltage trends from the API data
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_title('RMS Voltage (from API)')
ax.plot(pmon_df['time_us'], pmon_df['AVRMS'], 'k', label='A', alpha=0.7)
ax.plot(pmon_df['time_us'], pmon_df['BVRMS'], 'r', label='B', alpha=0.7)
ax.plot(pmon_df['time_us'], pmon_df['CVRMS'], 'b', label='C', alpha=0.7)
ax.set_ylabel('RMS Voltage (V)')
ax.set_xlabel('Time (UTC)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Query CPOW data

The `/api/v1/devices/{id}/cpow/data` endpoint returns raw waveform samples.

**CPOW requires an explicit `start_time`/`end_time` window** (unlike PMon, which
accepts a bare `limit`). Below we anchor a short window to the most recent PMon
timestamp, so the query lands on live data whenever it is flowing. You can also
pass any ISO 8601 UTC window directly, e.g.
`start_time='2026-06-17T01:30:00Z', end_time='2026-06-17T01:30:01Z'`.

**Scaling:** CPOW data may be returned as raw int32 ADC counts (requiring
`vscale`/`iscale` from metadata) or as pre-scaled floats (legacy format).
The code below handles both cases automatically.

In [ ]:
import pandas as pd

# CPOW needs an explicit time window. Anchor a short (~1 s) window to the latest
# PMon timestamp fetched above, so the query targets recent live data.
window_end = pd.Timestamp(pmon_df['time_us'].max())
if window_end.tzinfo is None:
    window_end = window_end.tz_localize('UTC')
window_start = window_end - pd.Timedelta(seconds=1)
fmt = '%Y-%m-%dT%H:%M:%S.%fZ'
params = {
    'start_time': window_start.tz_convert('UTC').strftime(fmt),
    'end_time': window_end.tz_convert('UTC').strftime(fmt),
    'limit': 3200,
}
print(f"CPOW window: {params['start_time']} -> {params['end_time']}")

cpow_table = client.get_arrow(f'/api/v1/devices/{device_id}/cpow/data', params=params)
cpow_df = cpow_table.to_pandas()
print(f"Received {len(cpow_df)} rows ({len(cpow_df) / 32000:.1f} seconds at 32 kHz)")
print(f"Columns: {list(cpow_df.columns)}")

# Determine scaling: int columns need vscale/iscale, float columns are pre-scaled.
is_int = str(cpow_table.schema.field('VA').type).startswith('int')
if is_int:
    # The API may include scaling metadata in the schema; if not, default to 1.0.
    # When querying from parquet files directly, see Tutorial 1 for scaling details.
    vscale = 1.0  # Replace with actual scale if available from device config
    iscale = 1.0
    print(f"Data is int type - scaling with vscale={vscale}, iscale={iscale}")
else:
    vscale = 1.0
    iscale = 1.0
    print("Data is float type (pre-scaled)")

In [ ]:
# Plot a 100 ms waveform slice
n_samples = min(3200, len(cpow_df))  # 100 ms at 32 kHz
time_ms = np.arange(n_samples) / 32.0  # Convert to milliseconds

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_title('Voltage Waveforms (from API)')
for col, color in [('VA', 'black'), ('VB', 'red'), ('VC', 'blue')]:
    if col in cpow_df.columns:
        ax.plot(time_ms, cpow_df[col].values[:n_samples], color=color, label=col, alpha=0.8)
ax.set_xlabel('Elapsed time (ms)')
ax.set_ylabel('Voltage')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Raw SQL queries

The `/api/v1/query/sql` endpoint accepts `SELECT` statements and executes them
against the DuckDB database. This is useful for custom aggregations, time-based
filtering, and cross-table joins.

> **Note:** Only `SELECT` statements are allowed. The server enforces a default
> limit of 30 rows; pass a higher `limit` if needed.

> **Tip:** Table names may vary depending on gateway configuration. Use
> `SHOW TABLES` via the SQL endpoint to see available tables before writing queries.

In [ ]:
# Example: average voltage per phase
result = client.query_sql(
    "SELECT AVG(AVRMS) as avg_va, AVG(BVRMS) as avg_vb, AVG(CVRMS) as avg_vc FROM pmon_data",
    device_id=device_id,
)
result

In [ ]:
# Example: frequency statistics
result = client.query_sql(
    "SELECT MIN(FREQ) as freq_min, MAX(FREQ) as freq_max, AVG(FREQ) as freq_avg, "
    "STDDEV(FREQ) as freq_std FROM pmon_data",
    device_id=device_id,
)
result

In [ ]:
# Example: hourly power totals
result = client.query_sql(
    "SELECT date_trunc('hour', time_us) as hour, "
    "AVG(AWATT + BWATT + CWATT) as avg_total_watts, "
    "COUNT(*) as samples "
    "FROM pmon_data GROUP BY hour ORDER BY hour",
    device_id=device_id,
    limit=100,
)
result

## 6. Working with events

Power quality events (voltage sags, swells, transients, etc.) are accessible via
the events API. You can also subscribe to a real-time event stream using
Server-Sent Events (SSE).

In [ ]:
# Fetch recent events
events = client.get_events(device_id=device_id, limit=10)
print(f"Found {len(events)} event(s)")
for evt in events[:5]:
    print(f"  {evt.get('event_type', 'unknown')} at {evt.get('timestamp_formatted', 'N/A')}")

In [ ]:
# SSE event stream (runs for a few seconds then stops)
# This connects to GET /api/v1/events/stream and prints events as they arrive.
import time

print("Listening for events (5 seconds)...")
try:
    resp = requests.get(
        f'{client.base_url}/api/v1/events/stream',
        params={'device_id': device_id},
        stream=True,
        timeout=6,
    )
    start = time.time()
    for line in resp.iter_lines(decode_unicode=True):
        if time.time() - start > 5:
            break
        if line and line.startswith('data:'):
            print(f"  Event: {line[5:].strip()}")
except requests.exceptions.ReadTimeout:
    pass
print("Done.")

## 7. List available files

You can query which data files are available on the gateway without accessing the filesystem directly.

In [ ]:
# List CPOW files
resp = requests.get(f'{client.base_url}/api/v1/devices/cpow/list_files')
cpow_files = resp.json()['files']
print(f"CPOW files: {len(cpow_files)}")
for f in cpow_files[-5:]:
    print(f"  {f}")

In [ ]:
# List PMon files
resp = requests.get(f'{client.base_url}/api/v1/devices/pmon/list_files')
pmon_files = resp.json()['files']
print(f"PMon files: {len(pmon_files)}")
for f in pmon_files[-5:]:
    print(f"  {f}")

## Next steps

- **Tutorial 4** (`04-live-streaming.ipynb`): Connect to live WebSocket streams for real-time spectral and waveform data.
- **Tutorial 2** (`02-local-duckdb.ipynb`): Run SQL queries directly against parquet files with DuckDB.
- **Tutorial 1** (`01-parquet-files.ipynb`): Access data directly from parquet files on the gateway filesystem.